In [2]:
import pandas as pd
import numpy as np

# === Datei laden ===
df = pd.read_csv("ParetoFront.csv")  # ggf. Pfad anpassen

# === Zielspalten definieren ===
all_objectives = [
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]

# === 1. Paretofront aus Transport Attachments & Attachments extrahieren ===
def pareto_front_2d(points):
    points = np.array(points)
    is_efficient = np.ones(points.shape[0], dtype=bool)
    for i, c in enumerate(points):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(points[is_efficient] < c, axis=1)
                | np.all(points[is_efficient] == c, axis=1)
            )
            is_efficient[i] = True
    return points[is_efficient]

pareto_points = pareto_front_2d(df[["Transport Attachments", "Attachments"]].values)
df_pareto_attach = pd.DataFrame(
    pareto_points, columns=["Transport Attachments", "Attachments"]
).drop_duplicates()

# Solution IDs der Pareto-optimalen Kombinationen ermitteln
pareto_attach_ids = df.merge(df_pareto_attach, on=["Transport Attachments", "Attachments"])[["Solution ID", "Transport Attachments", "Attachments"]]
print("\n🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):")
print(pareto_attach_ids.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === 2. Lösungen erweitern ===
df_expanded = df.drop(columns=["Transport Attachments", "Attachments"]).merge(
    pareto_attach_ids, how="cross"
)
# Kombiniere Solution ID_x und Solution ID_y zu einer neuen Spalte "Solution ID"
df_expanded["Solution ID"] = df_expanded["Solution ID_x"].astype(str) + "_" + df_expanded["Solution ID_y"].astype(str)

# Die neue "Solution ID" Spalte an den Anfang verschieben und die alten entfernen
cols = ["Solution ID"] + [col for col in df_expanded.columns if col not in ["Solution ID", "Solution ID_x", "Solution ID_y", "Solution ID_X_Y"]]
df_expanded = df_expanded[cols]

# === 3. Finaler Paretofilter (alle Ziele) ===
def pareto_filter_nd(df, objective_cols):
    values = df[objective_cols].values
    is_efficient = np.ones(values.shape[0], dtype=bool)
    for i, v in enumerate(values):
        if is_efficient[i]:
            is_efficient[is_efficient] = (
                np.any(values[is_efficient] < v, axis=1)
                | np.all(values[is_efficient] == v, axis=1)
            )
            is_efficient[i] = True
    return df[is_efficient].reset_index(drop=True)

df_final_pareto = pareto_filter_nd(df_expanded, all_objectives)

# Sortiere die Spalten in der gewünschten Reihenfolge
sort_cols = [
    "Solution ID",
    "Orders",
    "Order Items",
    "Driver Violation",
    "Commute Distance",
    "Transport Machines",
    "Transport Attachments",
    "Machines",
    "Workers",
    "Attachments"
]
df_final_pareto = df_final_pareto[sort_cols]

# === Insights ===
print("-"*40)
print("🔍 Insights zur Paretoanalyse\n" + "-"*40)
print(f"📦 Ursprüngliche Lösungen:         {len(df)}")
print(f"🎯 Pareto-Kombinationen (2D):      {len(df_pareto_attach)}")
print(f"🧩 Erweiterte Lösungskombis:       {len(df_expanded)}")
print(f"✅ Nicht-dominierte Endlösungen:   {len(df_final_pareto)}")
print(f"❌ Entfernte (dominierte) Lösungen: {len(df_expanded) - len(df_final_pareto)}\n")

print("📊 Pareto-Kombinationen (Anbaugeräte):")
print(df_pareto_attach.sort_values(["Transport Attachments", "Attachments"]).to_string(index=False))

# === Ergebnis speichern ===
df_final_pareto.to_csv("ParetoFront_filtered.csv", index=False)
print(f"\n💾 Datei gespeichert unter: ParetoFront_filtered.csv")



🆔 Solution IDs der Pareto-optimalen (Transport Attachments, Attachments):
 Solution ID  Transport Attachments  Attachments
         210                1462.17           57
         178                4932.71           44
----------------------------------------
🔍 Insights zur Paretoanalyse
----------------------------------------
📦 Ursprüngliche Lösungen:         979
🎯 Pareto-Kombinationen (2D):      2
🧩 Erweiterte Lösungskombis:       1958
✅ Nicht-dominierte Endlösungen:   38
❌ Entfernte (dominierte) Lösungen: 1920

📊 Pareto-Kombinationen (Anbaugeräte):
 Transport Attachments  Attachments
               1462.17         57.0
               4932.71         44.0

💾 Datei gespeichert unter: ParetoFront_filtered.csv


In [3]:
# Inhalt der gefilterten Paretofront ausgeben
print("\n📄 Inhalt der gefilterten Paretofront:")
print("-" * 40)
print(df_final_pareto.to_string(index=False))
print("-" * 40)
print("Lösungen:", len(df_final_pareto))



📄 Inhalt der gefilterten Paretofront:
----------------------------------------
Solution ID  Orders  Order Items  Driver Violation  Commute Distance  Transport Machines  Transport Attachments  Machines  Workers  Attachments
      1_178      85          884               365         323458.00            64794.07                4932.71        65      118           44
      1_210      85          884               365         323458.00            64794.07                1462.17        65      118           57
      2_178      85          884               366         322743.89            64829.81                4932.71        65      118           44
      2_210      85          884               366         322743.89            64829.81                1462.17        65      118           57
      3_178      85          884               367         322760.73            64814.37                4932.71        65      118           44
      3_210      85          884               367      

In [2]:
import json
import pandas as pd
from pathlib import Path

# === Pfade definieren ===
solutions_path = Path("pareto_solutions.json")
filtered_path = Path("ParetoFront_filtered.csv")
output_path = Path("pareto_solutions_filtered.json")

# === Dateien laden ===

# 1. JSON mit vollständigen Lösungen
with open(solutions_path, "r", encoding="utf-8") as f:
    solutions = json.load(f)

# 2. CSV mit kombinierten IDs (z. B. "1_178")
df_filtered = pd.read_csv(filtered_path)

# Sicherstellen, dass die relevante Spalte existiert
if "Solution ID" not in df_filtered.columns:
    raise ValueError("Die CSV-Datei muss eine Spalte 'Solution ID' enthalten.")

# === Neue kombinierte Lösungen erstellen ===
combined = {}

for _, row in df_filtered.iterrows():
    combined_id = row["Solution ID"]
    base_id, attach_id = combined_id.split("_")

    if base_id not in solutions or attach_id not in solutions:
        print(f"⚠️ ID {combined_id} enthält ungültige Referenz: {base_id} oder {attach_id} nicht gefunden.")
        continue

    base = solutions[base_id]
    attach = solutions[attach_id]

    combined[combined_id] = {
        "worker_route_plan": base["worker_route_plan"],
        "machine_route_plan": base["machine_route_plan"],
        "attachment_route_plan": attach["attachment_route_plan"],
        "combined_from": {
            "worker_machine_id": base_id,
            "attachment_id": attach_id
        }
    }

# === Neue Datei speichern ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print(f"✅ {len(combined)} kombinierte Lösungen gespeichert unter: {output_path}")

✅ 38 kombinierte Lösungen gespeichert unter: pareto_solutions_filtered.json


In [6]:
import json
import pandas as pd

# === Beispielhafte Einbindung der Datenstruktur (hier aus Datei oder Copy-Paste) ===
# Wenn du das JSON aus einer Datei laden möchtest, nutze:
# with open("machine_route_plan.json", "r") as f:
#     machine_route_plan = json.load(f)

# Direkt hier einfügen:
machine_route_plan = {
      "0": [
        427,
        387,
        306,
        207,
        511,
        486,
        901,
        700
      ],
      "1": [
        130,
        243,
        184,
        461,
        420,
        425,
        451,
        752,
        828,
        314,
        448,
        102,
        715
      ],
      "2": [
        661,
        358,
        97,
        924,
        370,
        322,
        587,
        576
      ],
      "3": [
        482,
        389,
        857,
        741,
        346,
        780,
        897,
        561
      ],
      "4": [
        64,
        589,
        542,
        810,
        89,
        390,
        802,
        248,
        797,
        624,
        821,
        241
      ],
      "5": [
        681,
        843,
        257,
        629,
        395,
        468,
        491,
        837,
        288,
        616,
        206,
        220,
        620,
        37,
        447
      ],
      "6": [
        304,
        600,
        848,
        708,
        32,
        621,
        684,
        579,
        453,
        710,
        502,
        871
      ],
      "7": [
        275,
        644,
        198,
        344,
        156,
        268,
        215,
        521,
        313,
        851,
        659,
        917
      ],
      "8": [
        523,
        854,
        160,
        115,
        563,
        76,
        856,
        526,
        299,
        192,
        818,
        530,
        716,
        591
      ],
      "9": [
        272,
        683,
        549,
        688,
        894,
        714,
        419,
        326,
        94
      ],
      "10": [
        582,
        418,
        903,
        730,
        330,
        408,
        618,
        104,
        889,
        383,
        479
      ],
      "11": [
        336,
        515,
        302,
        874,
        811,
        864,
        603,
        885,
        893,
        823,
        778,
        815,
        923,
        813,
        183,
        397,
        345,
        2
      ],
      "12": [
        800,
        5,
        678,
        236,
        913,
        323,
        816,
        635,
        748,
        571,
        782,
        463,
        290,
        111,
        503
      ],
      "13": [
        308,
        287,
        342,
        754,
        537,
        24,
        545,
        922,
        14,
        13
      ],
      "14": [
        69,
        291,
        785,
        205,
        877,
        743,
        822,
        186,
        154,
        72
      ],
      "15": [
        406,
        495,
        584,
        750,
        807,
        74,
        890,
        402,
        846,
        320,
        513,
        695,
        733,
        723
      ],
      "16": [
        775,
        732,
        113,
        181,
        690,
        809,
        441,
        238,
        588,
        369,
        674,
        706,
        870,
        185,
        527,
        103,
        880,
        33,
        325
      ],
      "17": [
        484,
        356,
        735,
        602,
        211,
        755,
        481,
        555,
        504,
        772,
        498,
        541,
        510
      ],
      "18": [
        25,
        559,
        878,
        867,
        478,
        440
      ],
      "19": [
        112,
        509,
        833,
        467,
        798,
        850,
        505,
        455,
        46,
        255,
        485,
        292,
        703
      ],
      "20": [
        875,
        519,
        136,
        108,
        175,
        311,
        565,
        827,
        835,
        836,
        721,
        888,
        861,
        925
      ],
      "21": [
        165,
        285,
        371,
        431,
        139,
        229,
        717,
        161,
        128
      ],
      "22": [
        7,
        96,
        197,
        132,
        303,
        432,
        119,
        718,
        497,
        647,
        546,
        271,
        258,
        348,
        380,
        347,
        745,
        91,
        801,
        227,
        176
      ],
      "23": [
        309,
        740,
        142,
        693,
        920,
        144,
        749,
        649,
        528,
        413
      ],
      "24": [
        410,
        366,
        817,
        100,
        237,
        614,
        193,
        724,
        458,
        711,
        434,
        713,
        256,
        720,
        82,
        728,
        916
      ],
      "25": [
        575,
        892,
        765,
        734,
        872,
        368,
        666,
        921,
        849,
        109,
        439,
        632,
        267,
        643,
        747,
        251,
        35
      ],
      "26": [
        376,
        855,
        232,
        162,
        756,
        568,
        373,
        488,
        405,
        858,
        496,
        660,
        819,
        353,
        340,
        601
      ],
      "27": [
        230,
        676,
        172,
        847,
        570,
        28,
        454,
        604,
        472,
        300,
        805,
        655,
        493
      ],
      "28": [
        538,
        517,
        377,
        699,
        709,
        844,
        466,
        339,
        814,
        535,
        859
      ],
      "29": [
        645,
        829,
        590,
        124,
        471,
        269,
        59,
        767,
        547,
        127,
        912,
        487,
        196,
        914,
        550,
        152,
        560,
        518
      ],
      "30": [
        762,
        283,
        642,
        657,
        338,
        633,
        638,
        429
      ],
      "31": [
        83,
        636,
        610,
        760,
        71,
        651,
        868,
        263,
        34,
        17,
        685,
        443,
        658,
        476,
        190,
        328,
        222,
        669,
        381
      ],
      "32": [
        577,
        21,
        412,
        593,
        90,
        218,
        31,
        78,
        910,
        225
      ],
      "33": [
        316,
        895,
        253,
        360,
        826,
        581,
        457,
        67,
        592,
        753,
        56,
        469,
        305,
        174,
        298,
        595,
        394,
        42,
        500,
        572,
        301,
        403,
        812,
        552
      ],
      "34": [
        640,
        121,
        149,
        95,
        492,
        254,
        4,
        332,
        473,
        145,
        239,
        692,
        671
      ],
      "35": [
        341,
        261,
        284,
        573,
        199,
        567,
        363,
        224,
        768,
        280,
        400,
        170,
        879,
        701,
        315,
        191,
        438
      ],
      "36": [
        135,
        704,
        386,
        456,
        182,
        726,
        792,
        834,
        554,
        906,
        522,
        293,
        329
      ],
      "37": [
        260,
        210,
        915,
        795,
        294,
        173,
        599,
        444,
        426,
        650,
        84,
        169,
        362,
        278,
        446
      ],
      "38": [
        594,
        155,
        514,
        318,
        235,
        677,
        884,
        60,
        761,
        384,
        168,
        630
      ],
      "39": [
        679,
        27,
        670,
        20,
        324,
        789,
        379
      ],
      "40": [
        598,
        689,
        883,
        40,
        887,
        729,
        653,
        911,
        321,
        16,
        845
      ],
      "41": [
        352,
        357,
        404,
        804,
        831,
        459,
        637,
        214,
        586,
        247,
        295,
        891,
        421,
        131,
        88,
        365
      ],
      "42": [
        788,
        918,
        281,
        276,
        223,
        840,
        673,
        615,
        265,
        63,
        367,
        349,
        423
      ],
      "43": [
        411,
        435,
        6,
        626,
        166,
        319,
        334,
        607,
        578,
        532,
        533,
        613
      ],
      "44": [
        262,
        61,
        163,
        866,
        529,
        164,
        609,
        680
      ],
      "45": [
        475,
        687,
        50,
        707,
        372,
        465,
        781,
        264,
        585,
        203,
        250
      ],
      "46": [
        499,
        317,
        525,
        428,
        274,
        806,
        627,
        359,
        757,
        531,
        736,
        544,
        769,
        204,
        217,
        259,
        507
      ],
      "47": [
        694,
        628,
        793,
        824,
        234,
        134,
        873,
        776,
        15,
        825,
        9,
        385,
        480,
        417,
        178,
        784
      ],
      "48": [
        737,
        391,
        41,
        201,
        114,
        758,
        490,
        774,
        583,
        151,
        746,
        664,
        146,
        39,
        188,
        116,
        122,
        75
      ],
      "49": [
        860,
        625,
        696,
        617,
        796,
        470,
        596,
        853,
        551,
        656,
        279
      ],
      "50": [
        219,
        907,
        296,
        416,
        58,
        213,
        705,
        414,
        779,
        452,
        87,
        118,
        246,
        331,
        691,
        1,
        289,
        399
      ],
      "51": [
        150,
        787,
        702,
        722,
        663,
        187,
        900,
        374,
        882,
        212,
        597,
        77,
        652,
        361,
        662,
        307,
        808,
        375,
        608,
        631
      ],
      "52": [
        464,
        786,
        68,
        839,
        286,
        424,
        343,
        449,
        799,
        430,
        38,
        574,
        388,
        297
      ],
      "53": [
        506,
        23,
        605,
        171,
        770,
        70,
        240,
        79,
        863,
        140,
        66
      ],
      "54": [
        312,
        120,
        908,
        252,
        12,
        667,
        886,
        354,
        355,
        783,
        392,
        129
      ],
      "55": [
        398,
        49,
        159,
        460,
        393,
        838,
        564,
        622,
        557,
        378,
        619,
        794,
        123,
        865,
        539,
        442,
        110,
        382,
        553
      ],
      "56": [
        202,
        231,
        869,
        898,
        244,
        739,
        0,
        862,
        249,
        195,
        543,
        233,
        462,
        742,
        106,
        712,
        611,
        107,
        612,
        226
      ],
      "57": [
        148,
        686,
        177,
        520,
        228,
        516,
        117,
        216,
        3,
        153,
        401,
        167,
        80,
        180,
        773,
        52
      ],
      "58": [
        474,
        93,
        665,
        909,
        350,
        566,
        409,
        277,
        437,
        396
      ],
      "59": [
        445,
        62,
        562,
        273,
        841,
        682,
        534,
        764,
        99,
        852,
        200,
        19
      ],
      "60": [
        483,
        639,
        905,
        45,
        137,
        57,
        266,
        126,
        751,
        771,
        919,
        876,
        44,
        548,
        634,
        351
      ],
      "61": [
        86,
        310,
        22,
        47,
        65,
        327,
        209,
        790,
        364,
        902,
        422,
        668,
        157,
        524,
        208,
        54,
        648,
        51,
        477
      ],
      "62": [
        194,
        26,
        270,
        832,
        55,
        147,
        842,
        81,
        896,
        494,
        30,
        29,
        245,
        731,
        179
      ],
      "63": [
        791,
        606,
        433,
        556,
        242,
        725,
        189,
        489,
        623,
        738,
        48,
        92,
        335,
        536
      ],
      "64": [
        85,
        675,
        580,
        777,
        558,
        904,
        508,
        407,
        698,
        830,
        697
      ]
    }

# === IDs aus allen Maschinenrouten sammeln ===
all_ids = []

for route in machine_route_plan.values():
    all_ids.extend(route)

# === Duplikate entfernen und sortieren ===
unique_sorted_ids = sorted(set(all_ids))

# === Fehlende IDs (Gaps) identifizieren ===
missing_ids = []

for i in range(len(unique_sorted_ids) - 1):
    current_id = unique_sorted_ids[i]
    next_id = unique_sorted_ids[i + 1]
    
    if next_id - current_id > 1:
        # IDs zwischen current_id und next_id sammeln (exklusiv)
        missing_range = list(range(current_id + 1, next_id))
        missing_ids.extend(missing_range)
        print(f"Lücke zwischen {current_id} und {next_id}: fehlt {missing_range}")

# === Gesamtübersicht ===
print(f"\n🔍 Insgesamt {len(missing_ids)} fehlende IDs:")
print(missing_ids)

Lücke zwischen 7 und 9: fehlt [8]
Lücke zwischen 9 und 12: fehlt [10, 11]
Lücke zwischen 17 und 19: fehlt [18]
Lücke zwischen 35 und 37: fehlt [36]
Lücke zwischen 42 und 44: fehlt [43]
Lücke zwischen 52 und 54: fehlt [53]
Lücke zwischen 72 und 74: fehlt [73]
Lücke zwischen 97 und 99: fehlt [98]
Lücke zwischen 100 und 102: fehlt [101]
Lücke zwischen 104 und 106: fehlt [105]
Lücke zwischen 124 und 126: fehlt [125]
Lücke zwischen 132 und 134: fehlt [133]
Lücke zwischen 137 und 139: fehlt [138]
Lücke zwischen 140 und 142: fehlt [141]
Lücke zwischen 142 und 144: fehlt [143]
Lücke zwischen 157 und 159: fehlt [158]
Lücke zwischen 220 und 222: fehlt [221]
Lücke zwischen 281 und 283: fehlt [282]
Lücke zwischen 332 und 334: fehlt [333]
Lücke zwischen 336 und 338: fehlt [337]
Lücke zwischen 414 und 416: fehlt [415]
Lücke zwischen 435 und 437: fehlt [436]
Lücke zwischen 449 und 451: fehlt [450]
Lücke zwischen 500 und 502: fehlt [501]
Lücke zwischen 511 und 513: fehlt [512]
Lücke zwischen 539 und 5